# 03 — Selected-model inference and local submission

Load a provisional or verified selection manifest, inspect predictions, and generate a uniquely named, structurally validated Kaggle CSV. No Kaggle upload occurs.

In [ ]:
from pathlib import Path
import sys
import os
import json
import numpy as np
import pandas as pd
import torch

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / 'src' / 'filament_seg' / 'submission.py').is_file() else cwd.parent
assert (project_root / 'src' / 'filament_seg' / 'submission.py').is_file(), cwd
if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

from filament_seg.utils import load_config, get_device
from filament_seg.benchmark import load_candidate, sha256_file, code_revision
from filament_seg.dataset import SolarFilamentDataset
from filament_seg.inference import predict_tiled, iter_probability_instances
from filament_seg.evaluate import diagnostic_panoptic_quality, evaluate_full_resolution_dataset
from filament_seg.submission import encode_coco_counts, decode_coco_counts, validate_submission_rows
if str(project_root) not in sys.path: sys.path.insert(0, str(project_root))
from scripts.generate_selected_submission import generate_from_manifest

device = get_device()
print(f'Python: {sys.executable}')
print(f'Device: {device}')
assert Path(sys.executable).resolve() == (project_root / '.conda' / 'python.exe').resolve(), sys.executable


## 1. Load the explicitly selected trained checkpoint

Set `FILAMENT_SELECTION_MANIFEST` when multiple selection manifests exist. Checkpoint and source hashes must match the selected manifest.

In [ ]:
manifest_paths = sorted((project_root / 'reports').glob('selected_model_*.json'))
manifest_override = os.environ.get('FILAMENT_SELECTION_MANIFEST')
if manifest_override:
    selection_manifest_path = Path(manifest_override).resolve()
elif len(manifest_paths) == 1:
    selection_manifest_path = manifest_paths[0]
else:
    raise ValueError('Set FILAMENT_SELECTION_MANIFEST to an explicit selected_model_*.json path')
selection = json.loads(selection_manifest_path.read_text(encoding='utf-8'))
ckpt_path = Path(selection['checkpoint'])
assert sha256_file(ckpt_path) == selection['checkpoint_sha256'], 'Checkpoint hash mismatch'
assert code_revision(project_root)['source_sha256'] == selection['code_revision']['source_sha256'], 'Source changed since selection'
model, model_metadata = load_candidate(ckpt_path, device=device)
assert model_metadata['architecture'] == selection['architecture']
print('Selection:', selection['status'], selection['experiment_id'])
print('Trained checkpoint:', ckpt_path, 'epoch:', model_metadata['epoch'])

## 2. Full-Resolution Tiled Inference with Hann Window Blending

Demonstrate seamless 2048x2048 prediction with 512x512 tiles and 128px cosine overlap blending.

In [ ]:
test_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026' / 'test' / 'test_images'
test_images = sorted(test_dir.glob('*.jpeg'))
assert len(test_images) == 180, f'Expected 180 test images, found {len(test_images)}'

sample_test = test_images[0]
import cv2
img = cv2.imread(str(sample_test), cv2.IMREAD_GRAYSCALE).astype(np.float32) / 255.0

prob_map = predict_tiled(model, img, tile_size=selection['tiling']['tile_size'], overlap=selection['tiling']['overlap'], batch_size=selection['tiling']['batch_size'], device=device)
print(f'Test image: {sample_test.name}')
print(f'Output shape: {prob_map.shape}, range: [{prob_map.min():.4f}, {prob_map.max():.4f}]')

## 3. Thresholding, Instance Extraction & Panoptic Quality (PQ)

Extract individual filament instances using historical baseline settings (threshold=0.75, min_area=300).

In [ ]:
train_dir = project_root / 'data' / 'filament-segmentation-2026' / 'MAGFiLO_1.0_Kaggle_2026' / 'train'
dataset = SolarFilamentDataset(train_dir / 'train_images', train_dir / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json')
indices = np.random.default_rng(42).permutation(len(dataset))
val_indices = indices[565:].tolist()

# Evaluate on sample validation image
val_sample = dataset[val_indices[0]]
val_prob = predict_tiled(model, val_sample['image'], tile_size=selection['tiling']['tile_size'], overlap=selection['tiling']['overlap'], batch_size=selection['tiling']['batch_size'], device=device)
gt_instances = [inst['mask'] for inst in dataset.iter_instances(val_indices[0], 0)]
pred_instances = list(iter_probability_instances(val_prob, threshold=selection['postprocessing']['threshold'], min_area=selection['postprocessing']['min_area']))

pq_res = diagnostic_panoptic_quality(gt_instances, pred_instances)
print(f'Sample Validation File: {val_sample["filename"]}')
print(f'Ground Truth Instances: {len(gt_instances)}, Predicted Instances: {len(pred_instances)}')
print(f'Diagnostic PQ: {pq_res["pq"]:.4f} (TP={pq_res["tp"]}, FP={pq_res["fp"]}, FN={pq_res["fn"]})')

## 4. Compressed COCO RLE Verification & Schema Conformance

Verify compressed COCO string counts round-trip encoding and structural schema validation.

In [ ]:
if pred_instances:
    sample_mask = pred_instances[0]
    counts = encode_coco_counts(sample_mask)
    decoded = decode_coco_counts(counts, sample_mask.shape)
    assert np.array_equal(decoded, sample_mask), 'RLE round trip failed!'
    print('COCO RLE Round-Trip: PASS')
    print(f'Encoded counts snippet: {counts[:40]}... (length: {len(counts)})')

## 5. Generate the selected-model submission

When enabled, write a new CSV and provenance file. The selected manifest determines architecture, checkpoint, preprocessing, tiling, and post-processing.

In [ ]:
RUN_SUBMISSION = False  # Set True to generate a new local artifact from the selected manifest.
if RUN_SUBMISSION:
    output_path, provenance_path = generate_from_manifest(selection_manifest_path)
    print('Validated CSV:', output_path)
    print('Provenance:', provenance_path)
else:
    print('Submission generation skipped; existing CSV files are untouched.')